# Notebook 02: Data Cleaning

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Overview

This notebook transforms the raw violations CSV into a clean, analysis-ready dataset. Steps include:
1. Standardizing column names and data types
2. Parsing dates and extracting temporal features
3. Mapping violation cities to canonical Boston neighborhood names
4. Dropping duplicates and irrelevant columns
5. Handling missing values
6. Saving the cleaned dataset to `data/processed/violations_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(RAW_DIR / 'violations.csv', low_memory=False)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head(2)

## 1. Standardize Column Names

We strip whitespace and convert all column names to lowercase snake_case for consistency.

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())

## 2. Parse Dates & Extract Temporal Features

The `status_dttm` column contains datetime strings. We parse it and extract `year`, `month`, and `day_of_week` for temporal analysis.

In [ ]:
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
df['year'] = df['status_dttm'].dt.year
df['month'] = df['status_dttm'].dt.month
df['day_of_week'] = df['status_dttm'].dt.day_name()

print('Records with valid date:', df['status_dttm'].notna().sum())
print('Records missing date:  ', df['status_dttm'].isna().sum())
print('Year range:', df['year'].min(), '–', df['year'].max())

## 3. Neighborhood Mapping

The `violation_city` field contains city/neighborhood names that are inconsistent (mixed case, abbreviations). We map them to canonical Boston neighborhood names. Records from outside Boston proper are flagged.

In [ ]:
# Show raw city values
print('Top 30 violation_city values:')
print(df['violation_city'].value_counts().head(30))

In [ ]:
neighborhood_map = {
    # Boston neighborhoods (direct)
    'Allston':           'Allston',
    'Back Bay':          'Back Bay',
    'Beacon Hill':       'Beacon Hill',
    'Brighton':          'Brighton',
    'Charlestown':       'Charlestown',
    'Chinatown':         'Chinatown',
    'Dorchester':        'Dorchester',
    'Downtown':          'Downtown',
    'East Boston':       'East Boston',
    'Fenway':            'Fenway',
    'Hyde Park':         'Hyde Park',
    'Jamaica Plain':     'Jamaica Plain',
    'Mattapan':          'Mattapan',
    'Mission Hill':      'Mission Hill',
    'North End':         'North End',
    'Roslindale':        'Roslindale',
    'Roxbury':           'Roxbury',
    'South Boston':      'South Boston',
    'South End':         'South End',
    'West End':          'West End',
    'West Roxbury':      'West Roxbury',
    # Aliases / common variants
    'Boston':            'Downtown',
    'E Boston':          'East Boston',
    'JP':                'Jamaica Plain',
    'So Boston':         'South Boston',
    'Roxbury Crossing':  'Roxbury',
    'Mattapan ':         'Mattapan',
    'Dorchester ':       'Dorchester',
    'BOSTON':            'Downtown',
    'DORCHESTER':        'Dorchester',
    'EAST BOSTON':       'East Boston',
    'ROXBURY':           'Roxbury',
    'MATTAPAN':          'Mattapan',
    'JAMAICA PLAIN':     'Jamaica Plain',
    'HYDE PARK':         'Hyde Park',
    'ROSLINDALE':        'Roslindale',
    'WEST ROXBURY':      'West Roxbury',
    'CHARLESTOWN':       'Charlestown',
    'BRIGHTON':          'Brighton',
    'ALLSTON':           'Allston',
    'SOUTH BOSTON':      'South Boston',
    'SOUTH END':         'South End',
}

# Apply case-insensitive mapping: normalize input then map
city_normalized = df['violation_city'].str.strip().str.title()
df['neighborhood'] = city_normalized.map(neighborhood_map)

# For unmapped entries, try title-cased pass-through if it's a known neighborhood
known_neighborhoods = set(neighborhood_map.values())
mask_unmapped = df['neighborhood'].isna()
df.loc[mask_unmapped, 'neighborhood'] = city_normalized[mask_unmapped].where(
    city_normalized[mask_unmapped].isin(known_neighborhoods)
)

mapped = df['neighborhood'].notna().sum()
total = len(df)
print(f'Mapped to neighborhood: {mapped:,} / {total:,} ({100*mapped/total:.1f}%)')
print('Neighborhood distribution:')
print(df['neighborhood'].value_counts())

## 4. Select & Rename Relevant Columns

We keep only the columns needed for analysis and rename them for clarity.

In [ ]:
keep_cols = [
    'case_no', 'status', 'status_dttm', 'year', 'month', 'day_of_week',
    'code', 'description', 'neighborhood',
    'violation_street', 'violation_zip', 'ward',
    'latitude', 'longitude',
]
df_clean = df[[c for c in keep_cols if c in df.columns]].copy()
print(f'Kept columns: {df_clean.columns.tolist()}')
print(f'Shape after column selection: {df_clean.shape}')

## 5. Handle Missing Values

- Records missing both `latitude`/`longitude` AND `neighborhood` are not spatially usable — we flag them.
- Records missing `status_dttm` retain their other fields and are included in non-temporal analyses.
- `description` NaN values are filled with `'Unknown'`.

In [ ]:
df_clean['description'] = df_clean['description'].fillna('Unknown')
df_clean['code'] = df_clean['code'].astype(str).str.strip()

# Convert lat/lon to numeric
df_clean['latitude'] = pd.to_numeric(df_clean['latitude'], errors='coerce')
df_clean['longitude'] = pd.to_numeric(df_clean['longitude'], errors='coerce')

# Flag records that have valid spatial data
df_clean['has_coords'] = df_clean['latitude'].notna() & df_clean['longitude'].notna()
df_clean['has_neighborhood'] = df_clean['neighborhood'].notna()

print('Records with coordinates:', df_clean['has_coords'].sum())
print('Records with neighborhood:', df_clean['has_neighborhood'].sum())

## 6. Remove Duplicates

We deduplicate on `case_no`, keeping the most recent record per case.

In [ ]:
before = len(df_clean)
df_clean = df_clean.sort_values('status_dttm', ascending=False)
df_clean = df_clean.drop_duplicates(subset='case_no', keep='first').reset_index(drop=True)
after = len(df_clean)
print(f'Removed {before - after:,} duplicate case numbers. Remaining: {after:,}')

## 7. Final Quality Check & Save

We verify the cleaned dataset and write it to `data/processed/violations_clean.csv`.

In [ ]:
print('=== Final Cleaned Dataset ===' )
print(f'Shape: {df_clean.shape}')
print(f'Date range: {df_clean["status_dttm"].min()} → {df_clean["status_dttm"].max()}')
print(f'Unique neighborhoods: {df_clean["neighborhood"].nunique()}')
print(f'Unique violation codes: {df_clean["code"].nunique()}')
print()
print('Missing values per column:')
print(df_clean.isnull().sum())

In [ ]:
out_path = PROCESSED_DIR / 'violations_clean.csv'
df_clean.to_csv(out_path, index=False)
print(f'Saved cleaned data → {out_path}')
df_clean.head(5)

## Summary

The cleaning pipeline:
- Standardized column names and data types
- Parsed dates and extracted year/month features
- Mapped `violation_city` values to canonical neighborhood labels
- Removed duplicate case numbers
- Flagged records with missing spatial information

The cleaned dataset is saved to `data/processed/violations_clean.csv` and will be the input for all downstream notebooks.

Proceed to **Notebook 03: EDA & Visualization**.